# PCA Morphospace

Principal components analysis of 21 selected features. Examines the dominant axes of
cross-sectional shape variation and evaluates within- vs between-individual spread.


## Setup

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path('../data')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

df = pd.read_csv(DATA_DIR / 'shape_analysis_results.csv')


## Feature selection (21 variables)

In [ ]:
SIZE_FEATURES  = ['area_mu2', 'min_mu', 'max_mu']
SHAPE_FEATURES = ['radial_mean_mu', 'circularity', 'eccentricity', 'aspect_ratio',
                  'solidity', 'convexity', 'radial_cv', 'n_radial_peaks',
                  'asymmetry_index', 'efd_deviation']
EFD_HARMONICS  = [f'efd_power_h{i}' for i in range(1, 9)]  # h1–h8 (h9,h10 near noise floor)
FEATURES = SIZE_FEATURES + SHAPE_FEATURES + EFD_HARMONICS
print(f'Feature set: {len(FEATURES)} variables')

X = df[FEATURES].dropna()
idx_clean = X.index
Xz = StandardScaler().fit_transform(X)
print(f'Samples after dropna: {len(X)}')


## PCA — variance explained

In [ ]:
pca = PCA(n_components=min(10, Xz.shape[1]))
scores = pca.fit_transform(Xz)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
evr = pca.explained_variance_ratio_ * 100
ax1.bar(range(1, len(evr)+1), evr, color='steelblue')
ax1.set_xlabel('PC'); ax1.set_ylabel('Variance explained (%)')
ax1.set_title('Scree plot')

ax2.plot(range(1, len(evr)+1), np.cumsum(evr), 'o-', color='steelblue')
ax2.axhline(80, color='crimson', linestyle='--', label='80%')
ax2.set_xlabel('PC'); ax2.set_ylabel('Cumulative variance (%)')
ax2.set_title('Cumulative variance'); ax2.legend()
plt.tight_layout(); plt.show()

for i, v in enumerate(evr[:5], 1):
    print(f'PC{i}: {v:.1f}%  (cumulative: {np.cumsum(evr)[i-1]:.1f}%)')


## PC1 vs PC2 scatter — coloured by individual

In [ ]:
meta = df.loc[idx_clean, 'mask_filename'].str.extract(r'^(\d+)_([AB])_(\d+)_mask')
meta.columns = ['individual', 'region', 'replicate']

individuals = meta['individual'].unique()
cmap = plt.cm.get_cmap('tab20', len(individuals))
ind_to_color = {ind: cmap(i) for i, ind in enumerate(individuals)}
colors = meta['individual'].map(ind_to_color)

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(scores[:, 0], scores[:, 1], c=colors, alpha=0.5, s=12, linewidths=0)
ax.set_xlabel(f'PC1 ({evr[0]:.1f}%)')
ax.set_ylabel(f'PC2 ({evr[1]:.1f}%)')
ax.set_title('Morphospace: PC1 vs PC2 (coloured by individual)')
plt.tight_layout(); plt.show()


## PC1 loadings — top contributors

In [ ]:
loadings = pd.Series(pca.components_[0], index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
colors_bar = ['crimson' if v > 0 else 'steelblue' for v in loadings]
ax.barh(loadings.index, loadings.values, color=colors_bar)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Loading')
ax.set_title('PC1 loadings')
plt.tight_layout(); plt.show()


## Within- vs between-individual spread

In [ ]:
df_pca = pd.DataFrame({'pc1': scores[:, 0], 'pc2': scores[:, 1],
                       'individual': meta['individual'].values}, index=idx_clean)

# Within-individual: mean spread (mean pairwise distance within each individual)
from scipy.spatial.distance import pdist
within = []
for ind, grp in df_pca.groupby('individual'):
    if len(grp) < 2:
        continue
    dists = pdist(grp[['pc1','pc2']].values)
    within.append(dists.mean())

# Between-individual: pairwise distance between individual centroids
centroids = df_pca.groupby('individual')[['pc1','pc2']].mean()
between = pdist(centroids.values).mean()
mean_within = np.mean(within)

print(f'Mean within-individual spread : {mean_within:.3f}')
print(f'Between-individual centroid spread: {between:.3f}')
print(f'Between/within ratio: {between/mean_within:.2f}')
